#### 랜덤포레스트
- 배깅 방식을 활용 
- 데이터 샘플링 방식 + 다수결|평균을 사용
- 사용하는 모델은 의사결정나무 고정 (랜덤포레스트 안에 의사결정나무 하드코딩)
- 각 노드에서 분할 할때마다 모든 피쳐에서 일부만 무작위 선택 -> 트리간의 상관성 낮춰준다. -> 다양성 증가 
- 여러 결정 트리를 서로 다르게 학습 시킨 뒤 다수결(분류), 평균(회귀)으로 예측하는 배깅 기반의 앙상블 
- 부트스트랩(중복 데이터를 허용)과 특성의 무작위 선택을 이용하여 모델간의 상관성을 낮춰서 성능과 안정성을 높이는 방법 
- 고차원 데이터에서 안정성이 굉장히 높다
- 대부분의 실무에서 많이 사용이 되는 알고리즘 

- parameter
    - n_estimators 
        - 기본값 : 100
        - 모델의 개수를 지정
    - criterion
        - 분류 : gini(기본값), entropy, log_loss
        - 회귀 : squared_error(기본값), absolute_error, feiedman_mse, poisson
    - max_depth
        - 기본값 : None
        - 트리의 최대 깊이
    - bootstrap
        - 기본값 : True
        - 트리 학습 데이터에 부트스트랩을 사용할것인가
    - max_features
        - 분류 : "sqrt" (피쳐의 개수의 루트 값)
        - 회귀 : 1.0 (모든 피쳐)
        - 가능한 값 : int(0.0 ~ 1.0), 'sqrt', 'log2'
    - min_sample_leaf
        - 기본값 : 1
        - 리프 노드의 최소 샘플의 수
        - 값이 큰 경우 -> 리프가 너무 세분화 되지 않도록 방지 (과적합 방지)
    - max_leaf_node
        - 기본값 : None
        - 트리 전체에서 리프 노드의 수를 제한 
        - 너무 많은 리프가 발생한다 -> 과적합
        - 너무 적은 리프가 발생한다 -> 모델의 단순화(성능 저하)
    - oob_score
        - 기본값 : False
        - 부트스트랩 사용시 빠진 샘플들을 이용하여 검증 점수를 출력
    - max_samples
        - 기본값 : None
        - 부트스트랩이 True인 경우, 각 트리가 사용할 샘플의 수(비율)
    - max_weight_fraction_leaf
        - 기본값 : 0.0
        - 샘플 가중치의 합이 전체 데이터에서 차지하는 비율 
        - 불균형 데이터셋에서 유용
    - class_weight
        - 기본값 : None
        - 데이터의 불균형에서 각각의 가중치를 부여할 수 있는 매개변수
        - 'balanced' : 소수의 데이터에 가중치를 추가적으로 부여하여 데이터 불균형 문제를 완화
        - dict 형태로 클래스별 가중치를 고정값으로 부여
- 속성
    - estimators_
        - 결정 트리들의 목록 
    - feature_importances_
        - 피쳐들의 중요도
    - oob_score
        - OOB를 이용한 검증 점수
    - oob_decision_function_
        - 분류 모델 사용 가능
        - OOB 데이터들의 확률 / 결정함수
    - oob_prediction_
        - 회귀 모델 사용 가능 
        - OOB 예측값
    - n_feature_in_ / feature_names_in_
        - 입력이 되는 피쳐의 개수  / 입력이 되는 피쳐들의 이름 목록
- 메서드
    - fit(x, y)
        - 모델에 학습 
    - predict(x)
        - 모델을 통한 예측
    - score(x, y)
        - 분류에서는 정확도, 회귀에서는 r2-score
    - apply(x)
        - 각 샘플이 각트리에서 도달하는 리프의 인덱스를 변환
    - decision_path(x)
        - 트리 내의 경로 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [2]:
body = pd.read_csv("../data/bodyPerformance.csv")
body.head()

,age,gender,height_cm,weight_kg,body fat_%,diastolic,systolic,gripForce,sit and bend forward_cm,sit-ups counts,broad jump_cm,class
0,27.0,M,172.3,75.24,21.3,80.0,130.0,54.9,18.4,60.0,217.0,C
1,25.0,M,165.0,55.80,15.7,77.0,126.0,36.4,16.3,53.0,229.0,A
2,31.0,M,179.6,78.00,20.1,92.0,152.0,44.8,12.0,49.0,181.0,C
3,32.0,M,174.5,71.10,18.4,76.0,147.0,41.4,15.2,53.0,219.0,B
4,28.0,M,173.8,67.70,17.1,70.0,127.0,43.5,27.1,45.0,217.0,B


In [3]:
body['gender'].map(
    lambda x: 0 if x == 'M' else 1
)

0        0
1        0
2        0
3        0
4        0
        ..
13388    0
13389    0
13390    0
13391    1
13392    0
Name: gender, Length: 13393, dtype: int64

In [4]:
np.where(
    body['gender'] == 'M', 0 , 1
)

array([0, 0, 0, ..., 0, 1, 0], shape=(13393,))

In [5]:
from sklearn.preprocessing import LabelEncoder

In [6]:
# LabelEncoder -> 문자형 데이터들을 숫자형으로 변환
# class 컬럼에서의 데이터는 A, B, C, D -> 1, 2, 3, 4 변환
# preprocessing --> fit() -> transform()

labelencoder = LabelEncoder()

In [7]:
body['class_1'] = labelencoder.fit_transform(body['class'])

In [8]:
body['class'].value_counts()

class
C    3349
D    3349
A    3348
B    3347
Name: count, dtype: int64

In [9]:
body['class_1'].value_counts()

class_1
2    3349
3    3349
0    3348
1    3347
Name: count, dtype: int64

In [10]:
body['gender'].value_counts()

gender
M    8467
F    4926
Name: count, dtype: int64

In [11]:
body['gender'] = labelencoder.fit_transform(body['gender'])

In [12]:
body['gender'].value_counts()

gender
1    8467
0    4926
Name: count, dtype: int64

In [13]:
body.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13393 entries, 0 to 13392
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      13393 non-null  float64
 1   gender                   13393 non-null  int64  
 2   height_cm                13393 non-null  float64
 3   weight_kg                13393 non-null  float64
 4   body fat_%               13393 non-null  float64
 5   diastolic                13393 non-null  float64
 6   systolic                 13393 non-null  float64
 7   gripForce                13393 non-null  float64
 8   sit and bend forward_cm  13393 non-null  float64
 9   sit-ups counts           13393 non-null  float64
 10  broad jump_cm            13393 non-null  float64
 11  class                    13393 non-null  object 
 12  class_1                  13393 non-null  int64  
dtypes: float64(10), int64(2), object(1)
memory usage: 1.3+ MB


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    body.drop(['class', 'class_1'], axis = 1), body['class_1'], test_size = 0.3, random_state = 42, stratify = body['class_1']
)

In [18]:
# 랜덤포레스트 모델을 생성 (기본값)
clf = RandomForestClassifier()

In [19]:
clf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
pred = clf.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.72      0.88      0.79      1004
           1       0.63      0.60      0.61      1004
           2       0.73      0.67      0.70      1005
           3       0.90      0.83      0.86      1005

    accuracy                           0.74      4018
   macro avg       0.74      0.74      0.74      4018
weighted avg       0.74      0.74      0.74      4018



In [24]:
# 모델의 개수를 증가
# 각 트리에서 사용이 되는 샘플의 수를 70%만 이용
# min_sample_leaf -> 4
clf2 = RandomForestClassifier(
    n_estimators = 200,
    max_samples = 1.0,
    min_samples_leaf = 1,
    max_features = 6
)
clf2.fit(X_train, y_train)
pred2 = clf2.predict(X_test)
print(classification_report(y_test, pred2))


              precision    recall  f1-score   support

           0       0.71      0.89      0.79      1004
           1       0.63      0.61      0.62      1004
           2       0.75      0.65      0.70      1005
           3       0.91      0.82      0.86      1005

    accuracy                           0.74      4018
   macro avg       0.75      0.74      0.74      4018
weighted avg       0.75      0.74      0.74      4018



In [25]:
credit = pd.read_csv('../data/credit_final.csv')
credit.head(3)

,credit.rating,account.balance,credit.duration.months,previous.credit.payment.status,credit.purpose,credit.amount,savings,employment.duration,installment.rate,marital.status,...,residence.duration,current.assets,age,other.credits,apartment.type,bank.credits,occupation,dependents,telephone,foreign.worker
0,1,1,18,3,2,1049,1,1,4,1,...,4,2,21,2,1,1,3,1,1,1
1,1,1,9,3,4,2799,1,2,2,3,...,2,1,36,2,1,2,3,2,1,1
2,1,2,12,2,4,841,2,3,2,1,...,4,1,23,2,1,1,2,1,1,1


In [26]:
credit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                          Non-Null Count  Dtype
---  ------                          --------------  -----
 0   credit.rating                   1000 non-null   int64
 1   account.balance                 1000 non-null   int64
 2   credit.duration.months          1000 non-null   int64
 3   previous.credit.payment.status  1000 non-null   int64
 4   credit.purpose                  1000 non-null   int64
 5   credit.amount                   1000 non-null   int64
 6   savings                         1000 non-null   int64
 7   employment.duration             1000 non-null   int64
 8   installment.rate                1000 non-null   int64
 9   marital.status                  1000 non-null   int64
 10  guarantor                       1000 non-null   int64
 11  residence.duration              1000 non-null   int64
 12  current.assets                  1000 non-null   int64
 13  age 

In [28]:
round(credit.describe(), 2)

,credit.rating,account.balance,credit.duration.months,previous.credit.payment.status,credit.purpose,credit.amount,savings,employment.duration,installment.rate,marital.status,...,residence.duration,current.assets,age,other.credits,apartment.type,bank.credits,occupation,dependents,telephone,foreign.worker
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,...,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,0.70,2.18,20.90,2.29,2.96,3271.25,1.87,2.45,2.97,2.37,...,2.84,2.36,35.54,1.81,1.93,1.37,2.90,1.16,1.40,1.04
std,0.46,0.84,12.06,0.62,0.97,2822.75,1.20,1.11,1.12,1.07,...,1.10,1.05,11.35,0.39,0.53,0.48,0.65,0.36,0.49,0.19
min,0.00,1.00,4.00,1.00,1.00,250.00,1.00,1.00,1.00,1.00,...,1.00,1.00,19.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,0.00,1.00,12.00,2.00,2.00,1365.50,1.00,2.00,2.00,1.00,...,2.00,1.00,27.00,2.00,2.00,1.00,3.00,1.00,1.00,1.00
50%,1.00,2.00,18.00,2.00,3.00,2319.50,1.00,2.00,3.00,3.00,...,3.00,2.00,33.00,2.00,2.00,1.00,3.00,1.00,1.00,1.00
75%,1.00,3.00,24.00,3.00,4.00,3972.25,3.00,4.00,4.00,3.00,...,4.00,3.00,42.00,2.00,2.00,2.00,3.00,1.00,2.00,1.00
max,1.00,3.00,72.00,3.00,4.00,18424.00,4.00,4.00,4.00,4.00,...,4.00,4.00,75.00,2.00,3.00,2.00,4.00,2.00,2.00,2.00


In [29]:
credit.columns

Index(['credit.rating', 'account.balance', 'credit.duration.months',
       'previous.credit.payment.status', 'credit.purpose', 'credit.amount',
       'savings', 'employment.duration', 'installment.rate', 'marital.status',
       'guarantor', 'residence.duration', 'current.assets', 'age',
       'other.credits', 'apartment.type', 'bank.credits', 'occupation',
       'dependents', 'telephone', 'foreign.worker'],
      dtype='object')

- credit.rating : 신용 상태 (0 : 불량 고객, 1 : 정상 고객) - target
- account.balance : 현재 계좌 잔고 (현재 계좌에 잔액에 따라 등급화)
- credit.duration.months : 대출 기간
- previous.credit.payment.status : 과거 대출 상환 이력
- credit.purpose : 대출 목적 (가전제품, 자동차, 교육비 등 대출 목적을 숫자로 인코딩)
- credit.amount :대출 금액
- employment.duration : 근속 연수
- installment.rate : 가처분 소득 대비 할부율
- marital.status : 결혼 상태 및 성별 ( 미혼 남성, 미혼 여성, 기혼 남성, 기혼 여성)
- guarantor : 보증인 여부
- residence.duration : 현 거주지 거주 기간
- current.assets : 보유 자산
- age : 나이
- other.credits : 타 기관 대출 여부
- apartment.type : 주거 형태 (자가, 전월세, 무상 거주)
- bank.credits : 당행 대출 건수 (이 은행에서의 진행 중인 대출 건수)
- occupation : 직업군 (무직, 단순 노무, 기술직 ...)
- dependents : 부양 가족 수
- telephone : 전화 번호 등록 여부 ( 본인 명의의 전화기 등록 여부)
- foreign.worker : 외국인 여부 (자국민, 외국인)

##### 연습 문제
1. credit에서 독립 변수와 종속 변수로 분할
2. train, test 셋으로 데이터를 분할 (8:2)
3. RF 모델을 생성(기본값)하여 모델의 성능을 평가
4. 더미 변수 생성
    - 컬럼은 4개
    - credit.purpose, apartment.type, occupation, marital.status
5. 데이터의 균형이 맞는가? 
    - 불균형하다면 샘플링 기법을 이용하여 데이터 균형을 맞춰준다. (오버샘플링 : 데이터의 개수가 너무 적기 때문, SMOTE)
    - 오버샘플링은 1:1
    - train 데이터만 샘플링 / test 데이터는 샘플링 x
6. 기본 RF 모델을 이용하여 성능을 평가

In [30]:
X = credit.drop('credit.rating', axis = 1)
y = credit['credit.rating']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

In [31]:
model = RandomForestClassifier()

In [35]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.78      0.52      0.62        60
           1       0.82      0.94      0.87       140

    accuracy                           0.81       200
   macro avg       0.80      0.73      0.75       200
weighted avg       0.81      0.81      0.80       200



In [32]:
df = pd.get_dummies(credit, columns = ['credit.purpose', 'apartment.type', 'occupation', 'marital.status'], drop_first = True)

In [38]:
df_X = df.drop('credit.rating', axis = 1)
df_y = df['credit.rating']

In [41]:
df_X_train, df_X_test, df_y_train, df_y_test = train_test_split(
    df_X, df_y, test_size=0.2, random_state=42, stratify=df_y
)

In [36]:
df['credit.rating'].value_counts()

credit.rating
1    700
0    300
Name: count, dtype: int64

In [42]:
from imblearn.over_sampling import SMOTE
smote = SMOTE()
sm_x, sm_y = smote.fit_resample(df_X_train, df_y_train)

In [43]:
model.fit(sm_x, sm_y)
pred2 = model.predict(df_X_test)
print(classification_report(df_y_test, pred2))

              precision    recall  f1-score   support

           0       0.60      0.58      0.59        60
           1       0.82      0.84      0.83       140

    accuracy                           0.76       200
   macro avg       0.71      0.71      0.71       200
weighted avg       0.76      0.76      0.76       200



In [44]:
clf2 = RandomForestClassifier(class_weight='balanced_subsample')
clf2.fit(X_train, y_train)
pred3 = clf2.predict(X_test)
print(classification_report(y_test, pred3))

              precision    recall  f1-score   support

           0       0.72      0.48      0.58        60
           1       0.81      0.92      0.86       140

    accuracy                           0.79       200
   macro avg       0.77      0.70      0.72       200
weighted avg       0.78      0.79      0.78       200



In [45]:
dummie_cols = ['credit.purpose', 'apartment.type', 'occupation', 'marital.status']
df_dummy = pd.get_dummies(credit, columns = dummie_cols, drop_first = True)

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    df_dummy.drop('credit.rating', axis = 1), df_dummy['credit.rating'],
    test_size = 0.2, random_state = 42, stratify = df_dummy['credit.rating']
)

In [47]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.72      0.47      0.57        60
           1       0.80      0.92      0.86       140

    accuracy                           0.79       200
   macro avg       0.76      0.69      0.71       200
weighted avg       0.78      0.79      0.77       200



In [48]:
clf2 = RandomForestClassifier(class_weight='balanced_subsample')
clf2.fit(X_train, y_train)
pred3 = clf2.predict(X_test)
print(classification_report(y_test, pred3))

              precision    recall  f1-score   support

           0       0.74      0.43      0.55        60
           1       0.79      0.94      0.86       140

    accuracy                           0.79       200
   macro avg       0.77      0.68      0.70       200
weighted avg       0.78      0.79      0.77       200



In [49]:
from imblearn.over_sampling import SMOTE
smote = SMOTE()

In [50]:
# train 데이터셋을 오버 샘플링
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [51]:
y_train_sm.value_counts()

credit.rating
1    560
0    560
Name: count, dtype: int64

In [52]:
y_train.value_counts()

credit.rating
1    560
0    240
Name: count, dtype: int64

In [53]:
model.fit(X_train_sm, y_train_sm)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [54]:
pred4 = model.predict(X_test)
print(classification_report(y_test, pred4))

              precision    recall  f1-score   support

           0       0.56      0.55      0.55        60
           1       0.81      0.81      0.81       140

    accuracy                           0.73       200
   macro avg       0.68      0.68      0.68       200
weighted avg       0.73      0.73      0.73       200



In [55]:
clf3 = RandomForestClassifier(n_estimators=300)
clf3.fit(X_train_sm, y_train_sm)
pred5 = clf3.predict(X_test)
print(classification_report(y_test, pred5))

              precision    recall  f1-score   support

           0       0.57      0.55      0.56        60
           1       0.81      0.82      0.82       140

    accuracy                           0.74       200
   macro avg       0.69      0.69      0.69       200
weighted avg       0.74      0.74      0.74       200



In [62]:
importance_df = pd.DataFrame(zip(X_train_sm.columns, clf3.feature_importances_), columns = ['feature_name', 'importance'])
importance_df.sort_values('importance', ascending = False).head(5)

,feature_name,importance
0,account.balance,0.167428
3,credit.amount,0.096170
1,credit.duration.months,0.089823
10,age,0.074510
4,savings,0.060344


In [56]:
car = pd.read_csv('../data/CarPrice_Assignment.csv')
df = car.select_dtypes('number')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   car_ID            205 non-null    int64  
 1   symboling         205 non-null    int64  
 2   wheelbase         205 non-null    float64
 3   carlength         205 non-null    float64
 4   carwidth          205 non-null    float64
 5   carheight         205 non-null    float64
 6   curbweight        205 non-null    int64  
 7   enginesize        205 non-null    int64  
 8   boreratio         205 non-null    float64
 9   stroke            205 non-null    float64
 10  compressionratio  205 non-null    float64
 11  horsepower        205 non-null    int64  
 12  peakrpm           205 non-null    int64  
 13  citympg           205 non-null    int64  
 14  highwaympg        205 non-null    int64  
 15  price             205 non-null    float64
dtypes: float64(8), int64(8)
memory usage: 25.8 K

In [57]:
df.drop('car_ID',axis = 1, inplace = True)

In [58]:
x = df.drop('price', axis = 1)
y = df['price']

In [59]:
reg = RandomForestRegressor(oob_score = True)

In [60]:
reg.fit(x, y)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


In [61]:
reg.oob_score_

0.9277813467540692